# LangChain: Agents & Tools

## Outline
* Agent concept and ReAct loop
* Build a tool with `@tool`
* Agent with built-in tools (Wikipedia, DuckDuckGo)
* Streaming agent output
* Tool error handling with middleware
* Dynamic system prompt


In [39]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")


from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)


In [ ]:
msg = [{"role": "user", "content": "What you can do?"}]
response = ollama.invoke(msg)
print(response.content)

## 1. Agent concept

Agent = LLM + Tools + Loop

**ReAct pattern:**
1. The LLM **thinks** (reasoning)
2. It **decides** which tool to use
3. It **executes** the tool
4. It **observes** the result and thinks again
5. It continues until it reaches the final answer

Old: `initialize_agent` + `AgentType`
New: `create_agent`


## 2. Build a Tool with `@tool`

In [44]:
from datetime import date
import requests
import xml.etree.ElementTree as ET

# Simple tool with decoration
@tool
def get_today_date(text: str) -> str:
    """Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string."""
    return str(date.today())

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input should be a valid Python math expression.
    Example: '2 + 2', '15 * 4', '100 / 5'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """
    Get weather for a city.

    IMPORTANT:
    Always pass city name in English (e.g. London, Tehran, Tokyo).
    Never use Persian or translated names.
    """    # In a real application, this should connect to an API
    weather_data = {
        "Tehran": "25°C, Sunny",
        "London": "15°C, Cloudy",
        "New York": "20°C, Partly cloudy",
        "Tokyo": "28°C, Humid",
    }
    return weather_data.get(city, f"Weather data for {city} not available.")


# Set user-agent for requests to arXiv
session = requests.Session()
session.headers.update({
    "User-Agent": "LF-ADP-Agent/1.0 (mailto:your.email@example.com)"
})

@tool
def arxiv_search_tool(query: str, max_results: int = 5) -> list[dict]:
    """
    Searches arXiv for research papers matching the given query.
    """
    url = f"https://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}"

    try:
        response = session.get(url, timeout=60)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return [{"error": str(e)}]

    try:
        root = ET.fromstring(response.content)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}

        results = []
        for entry in root.findall('atom:entry', ns):
            title = entry.find('atom:title', ns).text.strip()
            authors = [author.find('atom:name', ns).text for author in entry.findall('atom:author', ns)]
            published = entry.find('atom:published', ns).text[:10]
            url_abstract = entry.find('atom:id', ns).text
            summary = entry.find('atom:summary', ns).text.strip()

            link_pdf = None
            for link in entry.findall('atom:link', ns):
                if link.attrib.get('title') == 'pdf':
                    link_pdf = link.attrib.get('href')
                    break

            results.append({
                "title": title,
                "authors": authors,
                "published": published,
                "url": url_abstract,
                "summary": summary,
                "link_pdf": link_pdf
            })

        return results
    except Exception as e:
        return [{"error": f"Parsing failed: {str(e)}"}]

# Inspect tool metadata
print(f"Tool name: {get_today_date.name}")
print(f"Description: {get_today_date.description}")
print(f"Args: {get_today_date.args}")


Tool name: get_today_date
Description: Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string.
Args: {'text': {'title': 'Text', 'type': 'string'}}


In [45]:
# Agent with tools
agent = create_agent(
    model=llm,
    tools=[get_today_date, calculate, get_weather, arxiv_search_tool],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

# Agent with tools
agent_ollama = create_agent(
    model=ollama,
    tools=[get_today_date, calculate, get_weather, arxiv_search_tool],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is 25% of 300?"}]
})
print(response["messages"][-1].content)


25% of 300 is 75.


In [8]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the temperature in Tehran?"}]
})
print(response["messages"][-1].content)


The temperature in Tehran is 25°C and it is sunny.


In [10]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "show me papers about RL in finance?"}]
})
print(response["messages"][-1].content)


Here are some research papers related to Reinforcement Learning (RL) in finance:

1. **[FinRL: Deep Reinforcement Learning Framework to Automate Trading in Quantitative Finance](http://arxiv.org/abs/2111.09395v1)**
   - **Authors**: Xiao-Yang Liu, Hongyang Yang, Jiechao Gao, Christina Dan Wang
   - **Published**: 2021-11-07
   - **Summary**: This paper presents FinRL, an open-source framework designed to help quantitative traders automate trading using deep reinforcement learning. It features a modular architecture and provides various trading tasks as tutorials.
   - **[PDF Link](https://arxiv.org/pdf/2111.09395v1)**

2. **[FinRL-Podracer: High Performance and Scalable Deep Reinforcement Learning for Quantitative Finance](http://arxiv.org/abs/2111.05188v1)**
   - **Authors**: Zechu Li, Xiao-Yang Liu, Jiahao Zheng, Zhaoran Wang, Anwar Walid, Jian Guo
   - **Published**: 2021-11-07
   - **Summary**: This paper introduces the FinRL-Podracer framework, which accelerates the development of

In [21]:
response = agent_ollama.invoke({
    #"messages": [{"role": "user", "content": "use arxiv_search_tool tools to show me 2 papers and their details in arxiv about RL in finance?"}]
    "messages": [{"role": "user", "content": "show me 2 papers and their details in arxiv about RL in finance?"}]
})
print(response["messages"][-1].content)


Here are the 2 papers and their details in arXiv about RL in finance:

1. **FinRL: Deep Reinforcement Learning Framework to Automate Trading in Quantitative Finance**
	* Authors: Xiao-Yang Liu, Hongyang Yang, Jiechao Gao, Christina Dan Wang
	* Published: 2021-11-07
	* Summary: This paper presents a deep reinforcement learning framework called FinRL to automate trading in quantitative finance. FinRL is a full pipeline that helps quantitative traders overcome the steep learning curve in developing an agent that automatically positions to win in the market.
	* PDF: https://arxiv.org/pdf/2111.09395v1
2. **Quantum Game Theory in Finance**
	* Authors: Edward W. Piotrowski, J. Sladkowski
	* Published: 2004-06-18
	* Summary: This paper reviews the background and recent development in quantum game theory and its possible application in economics and finance. The intersection of science and society is also discussed.
	* PDF: https://arxiv.org/pdf/quant-ph/0406129v1


In [17]:
response = agent_ollama.invoke({
    "messages": [{"role": "user", "content": "What is the temperature in Tehran?"}]
})
print(response["messages"][-1].content)


The tool call response indicates that the current temperature in Tehran is 25°C, and the weather is sunny.


## 3. Built-in Tools — Wikipedia
``` pip install wikipedia ```

In [ ]:
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ✅ Set the user agent — without this, the Wikipedia API returns an empty response
wikipedia.set_user_agent("LangChain-Course-Bot/1.0 (educational purposes)")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1000,
    )
)

agent_wiki = create_agent(
    model=llm,
    tools=[wiki_tool, calculate, get_today_date],
    system_prompt="You are a research assistant. Use Wikipedia for factual questions.",
)

question = "Who invented Python programming language?"
response = agent_wiki.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


C:\Users\meisa\AppData\Local\Temp\ipykernel_37884\1024092919.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


Python programming language was invented by Guido van Rossum, who began working on it in the late 1980s as a successor to the ABC programming language.


In [26]:
agent_ollama_wiki = create_agent(
    model=ollama,
    tools=[wiki_tool, calculate, get_today_date],
    system_prompt="You are a research assistant. Use Wikipedia for factual questions.",
)

question = "Tell me the order in the resident evil game series based on wikipedia?"
response = agent_ollama_wiki.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)

The order of the Resident Evil game series based on Wikipedia is:

1. Resident Evil (1996)
2. Resident Evil 2 (1998)
3. Resident Evil 3: Nemesis (1999)
4. Resident Evil – Code: Veronica (2000)
5. Resident Evil 4 (2005)
6. Resident Evil 5 (2009)
7. Resident Evil 6 (2012)
8. Resident Evil 7: Biohazard (2017)
9. Resident Evil Village (2021)
10. Resident Evil Requiem (2026)

Note: The order may not include all games in the series, but it includes the mainline games in the correct order.


``` pip install ddgs```

In [30]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()

agent_ddg = create_agent(
    model=llm,
    tools=[ddg_search, calculate, get_today_date],
    system_prompt=(
        "You are a web search assistant. "
        "Use DuckDuckGo to find up-to-date information."
    ),
)

question = "Latest LangChain version 2026"
response = agent_ddg.invoke({"messages": [{"role": "user", "content": question}]})
print("\n=== DuckDuckGo Agent ===")
print(response["messages"][-1].content)



=== DuckDuckGo Agent ===
The latest version of LangChain as of August 7, 2026, is **langchain-openai==1.4.2**. This version includes updates and improvements, and it is part of a broader set of releases and product updates from LangChain. For more detailed information, you can check the release notes and changelogs on their official GitHub repository.


## 4. Streaming Agent Output
Streaming with stream_mode="values" — you can see each step


In [32]:
# Import message types from LangChain to distinguish between AI and user messages
from langchain.messages import AIMessage, HumanMessage

# Print header for the streaming output
print("=== Agent Streaming ===")

# Stream the agent's response in real-time chunks
# Each chunk represents a step in the agent's execution (thinking, tool calls, etc.)
for chunk in agent_ollama.stream(
    # Input query: user asks for weather information
    {"messages": [{"role": "user", "content": "What is the temperature in Tehran?"}]},
    stream_mode="values"  # Return the complete state after each step
):
    # Extract the most recent message from the conversation history
    latest = chunk["messages"][-1]
    
    # Check if the latest message is from the AI/agent
    if isinstance(latest, AIMessage):
        # Case 1: AI responded with text content
        if latest.content:
            print(f"[AI]: {latest.content}")
        # Case 2: AI wants to use a tool (function calling)
        elif latest.tool_calls:
            # Loop through all tool calls the AI wants to make
            for tc in latest.tool_calls:
                # Print which tool is being called with what arguments
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
    # Check if the message is a ToolMessage (result from a tool)
    # Tool messages have a 'name' attribute identifying which tool returned the result
    elif hasattr(latest, 'name'):  # ToolMessage
        # Print the first 100 characters of the tool's result
        print(f"[Tool Result]: {latest.content[:200]}")

=== Agent Streaming ===
[Tool Result]: What is the temperature in Tehran?
[Tool Call]: get_weather({'city': 'Tehran'})
[Tool Result]: 25°C, Sunny
[AI]: The current temperature in Tehran is 25°C, and the weather is sunny.


## 5. Tool Error Handling with Middleware

In [37]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Middleware for handling tool errors"""
    print("handle_tool_errors is called.")
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: {str(e)}. Please try a different approach.",
            tool_call_id=request.tool_call["id"]
        )

@tool
def risky_tool(number_a: str, number_b: str) -> str:
    """A tool that might fail. Input: two numbers. number_a / number_b"""
    return str(int(number_a) / int(number_b))  # division by zero if the value is 0

agent_safe = create_agent(
    #model=llm,
    model=ollama,
    tools=[risky_tool, calculate],
    middleware=[handle_tool_errors],
    system_prompt=(
        "You are a math assistant. "
        "ALWAYS use the risky_tool for ANY division operation, even if you think the result is undefined. "
        "Never answer math questions directly — always call the appropriate tool first."
    )
)
response = agent_safe.invoke({
    "messages": [{"role": "user", "content": "What is 10 divided by 0?"}]
})
print(response["messages"][-1].content)


handle_tool_errors is called.
Unfortunately, the risky_tool was unable to provide a result for this division operation. In mathematics, division by zero is undefined, and this is reflected in the tool's error message.
